In [85]:
#Imports
import pandas as pd
import numpy as np
import re
import string
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ( accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix)

In [86]:
#Load dataset
data = pd.read_csv("combined-set.csv")
data.head(1897)

,Unnamed: 0.1,Unnamed: 0,title,selftext,author,num_comments,is_suicide,url,selftext_clean,title_clean,author_clean,selftext_length,title_length,megatext_clean
0,0,339,Need help,Hi I don't really know how to phrase this situ...,F4GG0T_,6,0,https://www.reddit.com/r/depression/comments/f...,hi really know phrase situation try life reall...,need help,f 4 gg 0,2502,9,f 4 gg 0 hi really know phrase situation try l...
1,1,1386,feeling so overwhelmed and hopeless,i have been so depressed these past couple wee...,thiccbarbie420,2,1,https://www.reddit.com/r/SuicideWatch/comments...,depressed past couple week ever since got back...,feeling overwhelmed hopeless,th icc barbie 420,843,35,th icc barbie 420 depressed past couple week e...
2,2,544,"Nothing matters anymore, getting worse",Hi..I don't know where else to go. I am devast...,mjdg25,3,0,https://www.reddit.com/r/depression/comments/f...,hi know else go devastated right feeling like ...,nothing matter anymore getting worse,mj g 25,1909,38,mj g 25 hi know else go devastated right feeli...
3,3,937,Who’s tired of hearing bullshit,"The shit like “it will get better, everyone is...",fuckxboxandgames,4,1,https://www.reddit.com/r/SuicideWatch/comments...,shit like get better everyone purpose need fin...,tired hearing bullshit,fuck xbox game,403,31,fuck xbox game shit like get better everyone p...
4,4,741,I wish I was someone else.,I wish I was prettier. I wish I didn’t feel li...,Throwawaybluestahli,1,0,https://www.reddit.com/r/depression/comments/f...,wish wa prettier wish feel like burden wish br...,wish wa someone else,throwaway bluest ahli,321,26,throwaway bluest ahli wish wa prettier wish fe...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1890,374,619,think its over,i just don’t wanna live anymore so yeah,redignatz,0,0,https://www.reddit.com/r/depression/comments/f...,wanna live anymore yeah,think,red ign atz,39,14,red ign atz wanna live anymore yeah think
1891,375,141,To all of those feeling isolated and suffering...,I’ve learned that life is fucking sad sometime...,never-mind-la,0,0,https://www.reddit.com/r/depression/comments/f...,learned life fucking sad sometimes start think...,feeling isolated suffering right,never mind la,1123,59,never mind la learned life fucking sad sometim...
1892,376,998,I just really wish I had died the first time I...,That's all. Nothing has gotten better and I've...,WildButLoyalNugget,1,1,https://www.reddit.com/r/SuicideWatch/comments...,nothing ha gotten better tried hard gotten wor...,really wish died first time tried,wild loyal nugget,207,52,wild loyal nugget nothing ha gotten better tri...
1893,377,677,I feel unimportant.,Not the first time I'm going through this of c...,queessi,4,0,https://www.reddit.com/r/depression/comments/f...,first time going course recently hit really ha...,feel unimportant,que es,2490,19,que es first time going course recently hit re...


In [87]:
#Check columns
print(data.columns)
print(data.shape)

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'selftext', 'author',
       'num_comments', 'is_suicide', 'url', 'selftext_clean', 'title_clean',
       'author_clean', 'selftext_length', 'title_length', 'megatext_clean'],
      dtype='object')
(1895, 14)


In [88]:
#Keep only needed columns
data = data[['selftext_clean', 'is_suicide']].copy()
data.head()

,selftext_clean,is_suicide
0,hi really know phrase situation try life reall...,0
1,depressed past couple week ever since got back...,1
2,hi know else go devastated right feeling like ...,0
3,shit like get better everyone purpose need fin...,1
4,wish wa prettier wish feel like burden wish br...,0


In [89]:
#Removing Missing Values
data = data.dropna(subset=['selftext_clean', 'is_suicide']).copy()
print(data.shape)

(1894, 2)


In [90]:
#Clean labels properly
data['is_suicide'] = data['is_suicide'].astype(str).str.strip().str.lower()
data['is_suicide'].value_counts(dropna=False)
print(data.shape)

(1894, 2)


In [91]:
#Text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [92]:
#Create final text column
data['text'] = data['selftext_clean'].astype(str).apply(clean_text)
data[['selftext_clean', 'text']].head(3)

,selftext_clean,text
0,hi really know phrase situation try life reall...,hi really know phrase situation try life reall...
1,depressed past couple week ever since got back...,depressed past couple week ever since got back...
2,hi know else go devastated right feeling like ...,hi know else go devastated right feeling like ...


In [93]:
#Define text and labels
texts = data['text'].tolist()
labels = np.array(data['is_suicide'].tolist())

print("Unique labels:", np.unique(labels))
print("Number of texts:", len(texts))
print("Number of labels:", len(labels))

Unique labels: ['0' '1']
Number of texts: 1894
Number of labels: 1894


In [94]:
X_train_text, X_test_text, y_train, y_test = train_test_split(texts, labels,test_size=0.2,random_state=42,stratify=labels)

y_train = np.array(y_train).astype(int)
y_test = np.array(y_test).astype(int)

print("Train size:", len(X_train_text))
print("Test size:", len(X_test_text))
print("Train labels:", np.unique(y_train))
print("Test labels:", np.unique(y_test))

Train size: 1515
Test size: 379
Train labels: [0 1]
Test labels: [0 1]


In [95]:
#Term Frequency Inverse Document Frequency
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(1515, 20000)
(379, 20000)


In [96]:
y_train = np.array(y_train).astype(int)
y_test = np.array(y_test).astype(int)

print(type(y_train[0]), np.unique(y_train))
print(type(y_test[0]), np.unique(y_test))

<class 'numpy.int32'> [0 1]
<class 'numpy.int32'> [0 1]


In [97]:
#Model
# Model
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train_tfidf, y_train)

# Predictions
y_pred = model.predict(X_test_tfidf)
y_prob = model.predict_proba(X_test_tfidf)[:, 1]

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, zero_division=0))

# Report
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion Matrix
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.683377308707124
Precision: 0.7289156626506024
Recall: 0.6173469387755102
F1-score: 0.6685082872928176
              precision    recall  f1-score   support

           0       0.65      0.75      0.70       183
           1       0.73      0.62      0.67       196

    accuracy                           0.68       379
   macro avg       0.69      0.69      0.68       379
weighted avg       0.69      0.68      0.68       379

[[138  45]
 [ 75 121]]


In [98]:
import joblib

joblib.dump(model, "suicide_classifier_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Saved successfully.")

Saved successfully.


In [99]:
loaded_model = joblib.load("suicide_classifier_model.pkl")
loaded_tfidf = joblib.load("tfidf_vectorizer.pkl")

sample_text = "I feel hopeless and I do not want to live anymore"
sample_clean = clean_text(sample_text)
sample_vec = loaded_tfidf.transform([sample_clean])

pred = loaded_model.predict(sample_vec)[0]
prob = loaded_model.predict_proba(sample_vec)[0][1]

label_name = "suicide risk" if int(pred) == 1 else "non-suicide"

print("Prediction:", label_name)
print("Probability:", round(prob, 4))

Prediction: suicide risk
Probability: 0.6719


In [100]:
import json

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0))
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print(metrics)

{'accuracy': 0.683377308707124, 'precision': 0.7289156626506024, 'recall': 0.6173469387755102, 'f1_score': 0.6685082872928176}
